# Vapor-Eyes 05 — Synthesis: a shareable methane portfolio (PMTiles)

**Step 5: package the whole cascade into one shareable map.** NB01–04 produced the
regional screen, the confirmed plumes, and the operator attribution as Delta tables;
here we fold them into a single self-contained **vector PMTiles** archive anyone can
pan and zoom in a browser — no tile server:

- **Assemble** three layers — S5P hotspot cells (`h3_boundaryaswkb` hexagons), EMIT plume outlines, and TX RRC wells.
- **Encode + pyramid** each layer to Mapbox Vector Tiles across zoom levels with the `gbx_st_asmvt_pyramid` UDTF.
- **Fold** the whole pyramid into one PMTiles v3 archive with `gbx_pmtiles_agg` and view it inline.

**Result:** one `vapor_eyes.pmtiles` portfolio — screen → detect → quantify → attribute, in a single shareable map.

---
_Last Modified:_ July 11, 2026

![S5P hotspots + EMIT plumes + TX RRC wells → gbx_st_asmvt_pyramid per layer → gbx_pmtiles_agg → one vapor_eyes.pmtiles](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/vapor-eyes/vapor-eyes-05.png)

In [ ]:
%run ./config_nb

In [ ]:
MIN_Z, MAX_Z = 6, 13   # basin-to-local zoom range for the portfolio

## 1. Assemble the three portfolio layers

Each layer is a `(geom_wkb, attrs)` view over a cascade table: the S5P **hotspot** H3
cells as hexagons (Databricks-native `h3_boundaryaswkb`), the EMIT **plume** outlines
(`plume_geom`), and the TX RRC **wells** (`well_geom`). Run NB01–04 first.

In [ ]:
for _t in ("s5p_hotspots", "emit_plumes", "wells_shl"):
    assert spark.catalog.tableExists(_t), f"{_t} not found — run notebooks 01–04 first."

hotspots = spark.sql(
    "SELECT h3_boundaryaswkb(h3_cellid) AS geom_wkb, "
    "named_struct('ch4_max', ch4_max) AS attrs FROM s5p_hotspots"
)
plumes = spark.table("emit_plumes").select(
    F.col("plume_geom").alias("geom_wkb"),
    F.expr("named_struct('plume_id', plume_id, 'max_ppmm', max_conc_ppmm)").alias("attrs"),
)
wells = spark.table("wells_shl").select(
    F.col("well_geom").alias("geom_wkb"),
    F.expr("named_struct('operator', operator, 'api', api)").alias("attrs"),
)
print(f"... layers: hotspots={hotspots.count():,}  plumes={plumes.count():,}  wells={wells.count():,}")

## 2. Encode + pyramid each layer to vector tiles

`gbx_st_asmvt_pyramid` (a UDTF) emits one `(z, x, y, mvt_bytes)` row per feature per
zoom level, binning each geometry into the web-mercator tile grid and encoding
tile-local MVT with a **named layer**. We run it per layer and union the tile rows;
`gbx_pmtiles_agg` later merges same-tile features, preserving the three layers.

In [ ]:
def _pyramid(view_name, layer):
    return spark.sql(
        f"""
        SELECT t.z, t.x, t.y, t.mvt_bytes
        FROM {view_name},
             LATERAL gbx_st_asmvt_pyramid(geom_wkb, attrs, {MIN_Z}, {MAX_Z}, '{layer}') AS t
        """
    )

# a column key is required so the per-feature UDTF fans out on Serverless
hotspots.repartition(32, "geom_wkb").createOrReplaceTempView("_v_hotspots")
plumes.repartition(8, "geom_wkb").createOrReplaceTempView("_v_plumes")
wells.repartition(32, "geom_wkb").createOrReplaceTempView("_v_wells")

mvt = (
    _pyramid("_v_hotspots", "hotspots")
    .unionByName(_pyramid("_v_plumes", "plumes"))
    .unionByName(_pyramid("_v_wells", "wells"))
)
portfolio_mvt = finalize_delta(mvt, "portfolio_mvt_tiles", do_display=False)
_m = spark.table("portfolio_mvt_tiles")
print(f"portfolio_mvt_tiles: {_m.count():,} (z,x,y) MVT rows across z{MIN_Z}-z{MAX_Z}")
_m.groupBy("z").count().orderBy("z").limit(10).display()

## 3. Fold the pyramid into one PMTiles archive

`gbx_pmtiles_agg` folds all `(mvt_bytes, z, x, y)` rows into one PMTiles v3 archive
(BINARY), merging features that share a tile-id per layer. We write the single archive
to the Volume — a self-contained, shareable methane portfolio.

In [ ]:
import os  # noqa: E402

PORTFOLIO_PATH = f"{TILES_DIR}/vapor_eyes.pmtiles"
os.makedirs(TILES_DIR, exist_ok=True)  # Volume-FUSE-safe

if FORCE_REBUILD or not os.path.exists(PORTFOLIO_PATH):
    archive = (
        spark.table("portfolio_mvt_tiles")
        .groupBy(F.lit(1).alias("_g"))
        .agg(F.expr("gbx_pmtiles_agg(mvt_bytes, z, x, y)").alias("archive"))
        .select("archive")
        .collect()[0]["archive"]
    )
    with open(PORTFOLIO_PATH, "wb") as f:  # FUSE-safe sequential driver write
        f.write(archive)
    print(f"... wrote {PORTFOLIO_PATH} ({os.path.getsize(PORTFOLIO_PATH):,} bytes)")
else:
    print(f"... {PORTFOLIO_PATH} exists (skip; FORCE_REBUILD=False)")

## 4. View the methane portfolio inline

`show_pmtiles` prints the archive header, then renders through `INTERACTIVE_PLOTS`: a
static image by default (GitHub-friendly), or an interactive MapLibre map with all
three layers when `INTERACTIVE_PLOTS = True` — the archive base64-embedded in-browser.

In [ ]:
show_pmtiles(PORTFOLIO_PATH, emphasis="data")

## What we built — and the cascade

- **`portfolio_mvt_tiles`** (Delta) — the three-layer MVT pyramid (hotspots / plumes / wells).
- **`vapor_eyes.pmtiles`** — one self-contained vector archive: the whole cascade in a shareable map.

The **Vapor-Eyes** cascade end to end:
1. **Screen** — S5P TROPOMI methane → H3 hotspot cells (`netcdf_gbx`).
2. **Detect** — Sentinel-2 20 m SWIR proxy at the hotspot (`rst_mapalgebra`, `rst_h3_tessellate`).
3. **Quantify** — EMIT 60 m enhancement, clipped + summarized (`rst_clip`, `rst_summary`).
4. **Attribute** — nearest TX RRC candidate wells + operators (`st_distancesphere`).
5. **Synthesize** — one shareable PMTiles portfolio (`gbx_st_asmvt_pyramid`, `gbx_pmtiles_agg`).

GeoBrix: the **`netcdf_gbx`** reader, `EmitDownloader`/`TropomiDownloader`/`WellsDownloader`,
`rst_mapalgebra`, `rst_h3_tessellate`, `rst_clip`, `rst_summary`, `gbx_st_asmvt_pyramid`, `gbx_pmtiles_agg`.
Databricks-native: `st_geomfromwkb`, `st_point`, `st_x`/`st_y`, `st_distancesphere`, `h3_longlatash3`, `h3_boundaryaswkb`.